# spy-asian-pricer  —  Demo

Arithmetic-average Asian options on **SPY** under **Dupire local volatility**, calibrated from a JWSVI implied vol surface.

Pipeline: Yahoo Finance chains → SVI → JWSVI surface → Dupire LV → Monte Carlo (antithetic + geometric Asian CV) → Greeks.

Open this notebook in Colab via the badge in the README — the cell below will install everything you need.

In [ ]:
!pip install spy-asian-pricer[data,plot] --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from spy_asian_pricer import (
    calibrate_svi, JWSVIVolSurface, DupireLocalVol, AsianMCPricer,
    compute_greeks,
    check_butterfly_arbitrage, check_calendar_arbitrage, check_spread_arbitrage,
    fetch_spot, fetch_dividend_yield, fetch_risk_free_rate, build_vol_grid,
)

np.random.seed(42)


## 1. Pull market data and calibrate the surface

In [ ]:
spot = fetch_spot('SPY')
q = fetch_dividend_yield('SPY')              # ~1.3% for SPY (auto-normalized)
# Yahoo ^IRX (13-week T-bill) is the standard short-end proxy.  For longer
# Asian tenors pass tenor_years to interpolate up the Treasury curve.
r = fetch_risk_free_rate(tenor_years=0.5)
print(f'Spot ${spot:.2f}  |  r={r:.2%} (^IRX/^FVX interp)  |  q={q:.2%}  '
      f'->  fwd(1y) = ${spot * np.exp((r-q)):.2f}')

# build_vol_grid: IV is re-implied from lastPrice using these (r, q).
# Liquidity filter is `volume > 0` (any positive volume).  Yahoo's volume
# persists the previous trading session over weekends, so the filter is
# well-defined any time of week; OI is NOT used (Yahoo's OI is broken
# outside market hours).  No expiry-count cap.
vol_data = build_vol_grid('SPY', spot=spot, r=r, q=q)
print(f'{len(vol_data)} expiry slices')

jwsvi_slices = {}
for exp_str, df in vol_data.items():
    svi = calibrate_svi(df['logMoneyness'].values,
                        df['impliedVolatility'].values,
                        df['dcf'].iloc[0])
    jwsvi_slices[exp_str] = (svi.to_jwsvi(df['dcf'].iloc[0]), df['dcf'].iloc[0])

surface = JWSVIVolSurface(jwsvi_slices, spot=spot, r=r, q=q)
local_vol = DupireLocalVol(surface)            # inherits r, q from surface
print(f'\nDupire grid: K=[{local_vol.K_grid[0]:.1f}, {local_vol.K_grid[-1]:.1f}], '
      f'dcf=[{local_vol.dcf_grid[0]:.4f}, {local_vol.dcf_grid[-1]:.4f}]')
print('Dupire clamp diagnostics:')
for k, v in local_vol.clamp_stats.items():
    print(f'  {k:20s} {v}')


## 2. Visualise the implied vol surface

In [ ]:
K_plot = np.linspace(spot * 0.8, spot * 1.2, 80)
T_plot = np.linspace(surface.dcfs[0], surface.dcfs[-1], 60)
iv_grid = surface.implied_vol_grid(K_plot, T_plot)

fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(121, projection='3d')
K_mesh, T_mesh = np.meshgrid(K_plot, T_plot * 365)
ax1.plot_surface(K_mesh, T_mesh, iv_grid, cmap='viridis', alpha=0.8)
ax1.set_xlabel('Strike'); ax1.set_ylabel('DTE'); ax1.set_zlabel('IV')
ax1.set_title('Implied Vol Surface (JWSVI)')

ax2 = fig.add_subplot(122, projection='3d')
K_lv, T_lv = np.meshgrid(local_vol.K_grid, local_vol.dcf_grid * 365)
ax2.plot_surface(K_lv, T_lv, local_vol.lv_grid_data, cmap='inferno', alpha=0.8)
ax2.set_xlabel('Strike'); ax2.set_ylabel('DTE'); ax2.set_zlabel('Local Vol')
ax2.set_title('Dupire Local Vol Surface')
plt.tight_layout(); plt.show()

## 3. Arbitrage diagnostics

In [ ]:
K_chk = np.linspace(spot * 0.8, spot * 1.2, 80)
cal_ok, cal_n, _ = check_calendar_arbitrage(surface, K_chk)
print(f'Calendar arbitrage:  {"PASS" if cal_ok else f"FAIL ({cal_n})"}')

for exp_str, (jw, dcf) in list(jwsvi_slices.items())[:3]:
    svi = jw.to_svi(dcf)
    ok, n, gmin = check_butterfly_arbitrage(svi, dcf)
    sp_ok, sp_n, _ = check_spread_arbitrage(surface, dcf, K_chk, r)
    print(f'{exp_str}: butterfly {"PASS" if ok else f"FAIL ({n})"} (g_min={gmin:.2e})  |  spread {"PASS" if sp_ok else f"FAIL ({sp_n})"}')

## 4. Price an Asian call (antithetic-only MC)


In [ ]:
T = 0.5      # 6 months
n_obs = 126  # ~daily
K = spot     # ATM
n_paths = 200_000

pricer = AsianMCPricer(S0=spot, r=r, T=T, n_obs=n_obs,
                       vol_surface=surface, local_vol_surface=local_vol)

np.random.seed(42)
res = pricer.price_asian(K, n_paths)
print(f'Asian call (antithetic, no CV): ${res["price"]:.4f}  +/- ${res["std_err"]:.4f}')
print(f'  95% CI: [${res["ci_95"][0]:.4f}, ${res["ci_95"][1]:.4f}]   '
      f'paths={res["n_paths"]:,}')

# Geometric Asian Kemna-Vorst closed form, exposed for flat-vol benchmarking.
# (Not used as a control variate in price_asian -- see README "Why no CV?")
from spy_asian_pricer import geometric_asian_call_price
geom_kv = geometric_asian_call_price(spot, K, r, pricer.atm_iv, T, n_obs, q=q)
print(f'\nKemna-Vorst geometric Asian (flat ATM IV={pricer.atm_iv:.2%}): ${geom_kv:.4f}')
print(f'Arith vs geom gap: ${res["price"] - geom_kv:+.4f} '
      '(arith >= geom under any vol model)')


## 5. Greeks (finite difference under common random numbers)

In [ ]:
# Call greeks (q is taken from `surface` automatically).
g_call = compute_greeks(spot, K, r, T, n_obs, surface, local_vol,
                        n_paths=120_000, call=True)
g_put  = compute_greeks(spot, K, r, T, n_obs, surface, local_vol,
                        n_paths=120_000, call=False)

print(f'{"Greek":8s} {"Call":>10s} {"Put":>10s}')
for k in ['price', 'delta', 'gamma', 'vega', 'theta']:
    print(f'{k:8s} {g_call[k]:10.4f} {g_put[k]:10.4f}')

# Sign check + a Asian-flavored put-call parity look-up.
# (European-style identity: delta_call - delta_put = exp(-q*T).
#  Asian deltas only approximately satisfy this because of averaging.)
gap = (g_call['delta'] - g_put['delta']) - np.exp(-q * T)
print(f'\nDelta parity gap (Asian vs European exp(-qT)): {gap:+.4f}')


## 6. MC convergence: SE vs path count (antithetic only, expect $1/\sqrt{n}$)


In [ ]:
paths = [10_000, 25_000, 50_000, 100_000, 200_000, 500_000]
se = []
for n in paths:
    np.random.seed(42)
    se.append(pricer.price_asian(K, n)['std_err'])

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(paths, se, 'o-', label='Antithetic MC')
# Reference 1/sqrt(n) line, anchored to the first point.
ref = se[0] * np.sqrt(paths[0] / np.array(paths))
ax.loglog(paths, ref, 'k--', alpha=0.5, label=r'$\propto 1/\sqrt{n}$')
ax.set_xlabel('# paths'); ax.set_ylabel('Standard error ($)')
ax.set_title('Monte Carlo convergence (antithetic only)')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.show()
